# Release16 Nonconvex Checkpoint Objective Readiness v1

## Governance

- **Canonical baseline**: `soxlbacktest_qlib_walkforward_baseline_v1` and the current retained production composer as runtime context.
- **Direct comparison baseline**: `release16_risk_gated_state_head_objective_screen_v1`.
- **Role**: `diagnose` / training-objective handoff. This notebook does not promote or retain an actor.
- **Target window**: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- **Changed layer**: `model_selection`, `checkpoint_objective`, `state_head_target_gating`.
- **External reference**: repo-owned SOXLBacktest qlib baseline and state-objective target helpers.
- **Hypothesis**: Existing path-level candidates are insufficient; the next useful step is a fold-local nonconvex checkpoint objective that preserves risk-gated upside and keeps drawdown drift below the official replay gate.
- **Acceptance criteria**: target contract scope/lift/BC readiness pass, no eval leakage, and the objective recipe explicitly blocks actor retention until official three-fold replay clears `relative_return >= 30pp` and `MaxDD < 20%`.
- **Closest success**: `release16_risk_gated_state_head_objective_screen_v1` found a per-fold mix with all MaxDD under 20% and one passing fold.
- **Closest failure**: `release16_failedfold_state_head_checkpoint_bc_screen_v1` overfit/overexposed the failed folds and still failed all candidate gates.
- **Why this is not a duplicate**: this run converts the risk-gated per-fold near miss into a fold-local training/checkpoint objective with target-scope and label-readiness gates; it does not add another path-level threshold grid.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
for candidate_root in [ROOT, *ROOT.parents]:
    if (candidate_root / "SPEC.md").exists():
        ROOT = candidate_root
        break
else:
    raise RuntimeError("Could not locate repository root from notebook cwd")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_training_targets import (
    audit_state_objective_target_lift,
    audit_state_objective_target_scope,
    score_state_objective_checkpoint_candidates,
    summarize_state_objective_bc_readiness,
    summarize_state_objective_checkpoint_selection,
    summarize_state_objective_target_lift_audit,
    summarize_state_objective_target_scope_audit,
)

EXPERIMENT = "release16_nonconvex_checkpoint_objective_readiness_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
SOURCE_CARD = ROOT / "runtime/model_selection/card/release16_risk_gated_state_head_objective_screen_v1"
TARGET_CARD = ROOT / "runtime/model_selection/card/release16_failedfold_state_head_target_recipe_v1"
BC_SCREEN_CARD = ROOT / "runtime/model_selection/card/release16_failedfold_state_head_checkpoint_bc_screen_v1"
CARD.mkdir(parents=True, exist_ok=True)

REQUIRED_FOLDS = ["wf_2020", "wf_2022", "wf_2023_2024"]
FAILED_FOLDS = ["wf_2020", "wf_2023_2024"]
FOLD_WEIGHTS = {"wf_2020": 1.0, "wf_2022": 1.25, "wf_2023_2024": 1.0}
TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
DRAWDOWN_GAP_WEIGHT = 2.0

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))

source_summary = read_json(SOURCE_CARD / "summary.json")
source_selection = read_json(SOURCE_CARD / "selection.json")
target_summary = read_json(TARGET_CARD / "summary.json")
bc_summary_source = read_json(BC_SCREEN_CARD / "summary.json")

candidate_grid = pd.read_csv(SOURCE_CARD / "candidate_grid.csv")
selected_by_fold_source = pd.read_csv(SOURCE_CARD / "selected_by_fold.csv")
target_contract = pd.read_csv(TARGET_CARD / "state_head_target_contract.csv")
bc_readiness_scores = pd.read_csv(TARGET_CARD / "bc_readiness_scores.csv")
bc_slice_metrics = pd.read_csv(BC_SCREEN_CARD / "bc_slice_metrics.csv")

print({
    "candidate_rows": int(len(candidate_grid)),
    "target_contract_rows": int(len(target_contract)),
    "source_decision": source_summary.get("decision"),
    "source_gate_pass_count": source_summary.get("gate_pass_count"),
})

{'candidate_rows': 195, 'target_contract_rows': 3731, 'source_decision': 'risk_gated_state_head_objective_screen_no_full_gate_no_actor_retention', 'source_gate_pass_count': 1}


In [2]:
fold_defs = (
    target_contract[["fold", "train_start", "train_end", "eval_start", "eval_end"]]
    .drop_duplicates()
    .rename(columns={"fold": "name"})
    .sort_values("name")
    .to_dict(orient="records")
)

scope_audit = audit_state_objective_target_scope(target_contract, folds=fold_defs)
scope_summary = summarize_state_objective_target_scope_audit(scope_audit)

lift_audit = audit_state_objective_target_lift(
    target_contract,
    baseline_risk_position_col="risk_target_position_original",
)
lift_summary = summarize_state_objective_target_lift_audit(lift_audit)

bc_readiness_summary = summarize_state_objective_bc_readiness(bc_readiness_scores)

target_contract_gate = bool(scope_summary.get("training_allowed")) and bool(lift_summary.get("training_allowed"))
bc_readiness_gate = bool(bc_readiness_summary.get("selected_bc_readiness_gate"))

scope_audit.to_csv(CARD / "target_scope_audit.csv", index=False)
lift_audit.to_csv(CARD / "target_lift_audit.csv", index=False)
bc_readiness_scores.to_csv(CARD / "bc_readiness_scores.csv", index=False)

pd.DataFrame([
    {"gate": "target_scope", **scope_summary},
    {"gate": "target_lift", **lift_summary},
    {"gate": "bc_readiness", **bc_readiness_summary},
]).to_json(CARD / "target_readiness_summary.json", orient="records", indent=2)

print({
    "scope_decision": scope_summary.get("decision"),
    "lift_decision": lift_summary.get("decision"),
    "bc_decision": bc_readiness_summary.get("decision"),
    "target_contract_gate": target_contract_gate,
    "bc_readiness_gate": bc_readiness_gate,
})

{'scope_decision': 'state_objective_targets_training_ready', 'lift_decision': 'state_objective_target_lift_training_ready', 'bc_decision': 'state_objective_bc_ready_for_training', 'target_contract_gate': True, 'bc_readiness_gate': True}


In [3]:
grid = candidate_grid.copy()
grid["relative_gap_pp"] = (TARGET_RELATIVE_RETURN_PP - pd.to_numeric(grid["relative_return_pct"], errors="coerce")).clip(lower=0.0)
grid["drawdown_gap_pp"] = (pd.to_numeric(grid["max_dd_pct"], errors="coerce") - TARGET_MAX_DD_PCT).clip(lower=0.0)
grid["window_weight"] = grid["fold"].map(FOLD_WEIGHTS).fillna(1.0)
grid["weighted_window_gate_loss"] = grid["window_weight"] * (
    grid["relative_gap_pp"] + DRAWDOWN_GAP_WEIGHT * grid["drawdown_gap_pp"]
)
grid["candidate_fold_gate"] = (grid["relative_gap_pp"] == 0.0) & (grid["drawdown_gap_pp"] == 0.0)

global_rows = []
for candidate, group in grid.groupby("candidate_id", sort=True):
    present_folds = sorted(group["fold"].dropna().astype(str).unique().tolist())
    missing = sorted(set(REQUIRED_FOLDS) - set(present_folds))
    if missing:
        continue
    family = str(group["family"].iloc[0])
    weighted_gate_loss = float(group["weighted_window_gate_loss"].sum())
    max_relative_gap = float(group["relative_gap_pp"].max())
    max_drawdown_gap = float(group["drawdown_gap_pp"].max())
    gate_pass_count = int(group["candidate_fold_gate"].sum())
    global_rows.append({
        "candidate": str(candidate),
        "family": family,
        "fold_count": int(len(present_folds)),
        "gate_pass_count": gate_pass_count,
        "window_gate": bool(gate_pass_count == len(REQUIRED_FOLDS)),
        "coverage_gate": bool(target_contract_gate and bc_readiness_gate),
        "alignment_loss": weighted_gate_loss / 100.0,
        "window_gate_loss": weighted_gate_loss,
        "missing_target_ratio": 0.0 if target_contract_gate else 1.0,
        "rebound_position_mae": 0.0,
        "risk_cap_violation": max_drawdown_gap / 100.0,
        "score": 0.0,
        "min_relative_return_pp": float(group["relative_return_pct"].min()),
        "max_drawdown_pct": float(group["max_dd_pct"].max()),
        "max_relative_gap_pp": max_relative_gap,
        "max_drawdown_gap_pp": max_drawdown_gap,
        "mean_avg_pos": float(group["avg_pos"].mean()),
    })

global_alignment_scores = pd.DataFrame(global_rows)
checkpoint_objective_scores = score_state_objective_checkpoint_candidates(global_alignment_scores)
checkpoint_selection_summary = summarize_state_objective_checkpoint_selection(checkpoint_objective_scores)

candidate_gap_by_fold = grid.sort_values(
    ["candidate_fold_gate", "weighted_window_gate_loss", "candidate_id", "fold"],
    ascending=[False, True, True, True],
)
candidate_gap_by_fold.to_csv(CARD / "candidate_gap_by_fold.csv", index=False)
global_alignment_scores.to_csv(CARD / "global_candidate_objective_scores.csv", index=False)
checkpoint_objective_scores.to_csv(CARD / "checkpoint_objective_scores.csv", index=False)

best_existing_single_candidate = checkpoint_objective_scores.iloc[0].to_dict() if not checkpoint_objective_scores.empty else {}
print({
    "global_candidates": int(len(checkpoint_objective_scores)),
    "full_gate_existing_candidates": int(checkpoint_objective_scores["window_gate"].astype(bool).sum()) if not checkpoint_objective_scores.empty else 0,
    "best_existing_single_candidate": best_existing_single_candidate.get("candidate", ""),
    "best_single_min_relative": best_existing_single_candidate.get("min_relative_return_pp"),
})

{'global_candidates': 65, 'full_gate_existing_candidates': 0, 'best_existing_single_candidate': 'position_cap_0.20', 'best_single_min_relative': 11.420582}


In [4]:
selected_by_fold = source_summary.get("selected_by_fold", {})
selected_metrics = []
for fold in REQUIRED_FOLDS:
    row = dict(selected_by_fold.get(fold, {}))
    selected_metrics.append({
        "window": fold,
        "split_type": "walk_forward_test",
        "return_pct": float(row.get("selected_return", 0.0)),
        "relative_return_pct": float(row.get("selected_relative_return", 0.0)),
        "max_dd_pct": float(row.get("selected_max_dd", 0.0)),
        "avg_pos": float(row.get("selected_avg_pos", 0.0)),
        "candidate": str(row.get("selected_candidate", "")),
        "family": str(row.get("family", "")),
        "candidate_gate": bool(row.get("selected_candidate_gate", False)),
    })
selected_metrics_frame = pd.DataFrame(selected_metrics)
selected_metrics_frame.to_csv(CARD / "selected_fold_teacher_metrics.csv", index=False)

slice_required = bc_slice_metrics[bc_slice_metrics["slice"].isin([
    "failedfold_state_head_target_state",
    "is_downtrend_state",
    "is_rebound_cluster",
    "release_gate_state",
])].copy()
slice_required["accuracy"] = pd.to_numeric(slice_required["accuracy"], errors="coerce")
slice_gate_by_fold = (
    slice_required.groupby("fold", dropna=False)
    .agg(
        required_slice_count=("slice", "count"),
        min_required_accuracy=("accuracy", "min"),
        failed_slice_count=("accuracy", lambda s: int((s.fillna(0.0) < 0.60).sum())),
    )
    .reset_index()
)
slice_gate_by_fold["slice_precondition_gate"] = slice_gate_by_fold["failed_slice_count"] == 0
slice_gate_by_fold.to_csv(CARD / "state_head_slice_precondition.csv", index=False)

state_head_slice_precondition_gate = bool(slice_gate_by_fold["slice_precondition_gate"].all()) if not slice_gate_by_fold.empty else False
checkpoint_objective_allowed_next = bool(target_contract_gate and bc_readiness_gate)
ppo_allowed_next = bool(checkpoint_objective_allowed_next and state_head_slice_precondition_gate)
actor_retention_allowed = False
promotion_allowed = False

precondition_gates = {
    "target_contract_gate": target_contract_gate,
    "scope_leakage_gate": bool(scope_summary.get("leakage_gate")),
    "target_lift_gate": bool(lift_summary.get("lift_semantics_gate")),
    "bc_readiness_gate": bc_readiness_gate,
    "state_head_slice_precondition_gate": state_head_slice_precondition_gate,
    "checkpoint_objective_allowed_next": checkpoint_objective_allowed_next,
    "ppo_allowed_next": ppo_allowed_next,
    "promotion_allowed": promotion_allowed,
    "actor_retention_allowed": actor_retention_allowed,
    "full_gate_existing_candidate_count": int(checkpoint_objective_scores["window_gate"].astype(bool).sum()) if not checkpoint_objective_scores.empty else 0,
}
write_json(CARD / "precondition_gates.json", precondition_gates)

print(precondition_gates)

{'target_contract_gate': True, 'scope_leakage_gate': True, 'target_lift_gate': True, 'bc_readiness_gate': True, 'state_head_slice_precondition_gate': False, 'checkpoint_objective_allowed_next': True, 'ppo_allowed_next': False, 'promotion_allowed': False, 'actor_retention_allowed': False, 'full_gate_existing_candidate_count': 0}


In [5]:
objective_recipe = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_card": "release16_risk_gated_state_head_objective_screen_v1",
    "source_target_contract": "release16_failedfold_state_head_target_recipe_v1",
    "source_checkpoint_screen": "release16_failedfold_state_head_checkpoint_bc_screen_v1",
    "objective_name": "nonconvex_risk_gated_state_head_checkpoint_objective",
    "objective_terms": [
        {
            "term": "fold_weighted_relative_return_gap",
            "applies_to": REQUIRED_FOLDS,
            "formula": "sum(window_weight * max(0, 30 - relative_return_pp))",
            "purpose": "avoid the current single-candidate underparticipation failure",
        },
        {
            "term": "drawdown_drift_stop_loss",
            "applies_to": REQUIRED_FOLDS,
            "formula": "sum(window_weight * 2.0 * max(0, max_dd_pct - 20))",
            "purpose": "reject checkpoints that buy upside by breaching official replay drawdown",
        },
        {
            "term": "fold_local_state_head_target_alignment",
            "applies_to": ["wf_2020_train", "wf_2022_train", "wf_2023_2024_train"],
            "formula": "weighted BC/checkpoint loss over risk_target_position and rebound_target_position",
            "purpose": "make rulebase-reversal state-head lift learnable before official replay",
        },
        {
            "term": "state_head_slice_fit_guard",
            "applies_to": REQUIRED_FOLDS,
            "formula": "block PPO continuation until required sparse slices pass fit gates",
            "purpose": "prevent another actor from passing global fit while missing downtrend/release slices",
        },
    ],
    "target_contract": {
        "rows": int(len(target_contract)),
        "scope_summary": scope_summary,
        "lift_summary": lift_summary,
        "bc_readiness_summary": bc_readiness_summary,
    },
    "risk_gated_existing_path_readout": {
        "per_fold_teacher_min_relative_return_pp": float(selected_metrics_frame["relative_return_pct"].min()),
        "per_fold_teacher_worst_max_dd_pct": float(selected_metrics_frame["max_dd_pct"].max()),
        "best_existing_single_candidate": best_existing_single_candidate,
        "existing_full_gate_candidate_count": int(checkpoint_objective_scores["window_gate"].astype(bool).sum()) if not checkpoint_objective_scores.empty else 0,
    },
    "acceptance_gate": {
        "official_replay_required_folds": REQUIRED_FOLDS,
        "relative_return_pp_min": TARGET_RELATIVE_RETURN_PP,
        "max_dd_pct_strict_lt": TARGET_MAX_DD_PCT,
        "actor_retention_before_replay": False,
    },
    "precondition_gates": precondition_gates,
}
write_json(CARD / "objective_recipe.json", objective_recipe)

pd.DataFrame(objective_recipe["objective_terms"]).to_csv(CARD / "objective_terms.csv", index=False)
print(objective_recipe["objective_name"])

nonconvex_risk_gated_state_head_checkpoint_objective


In [6]:
decision = (
    "release16_nonconvex_checkpoint_objective_readiness_ready_no_actor_retention"
    if checkpoint_objective_allowed_next
    else "release16_nonconvex_checkpoint_objective_readiness_blocked_no_training"
)
next_mechanism = (
    "train_nonconvex_risk_gated_state_head_checkpoint_objective_with_slice_guard_then_official_three_fold_replay"
    if checkpoint_objective_allowed_next
    else "regenerate_fold_local_state_head_target_contract_before_training"
)

sidecars = {
    "target_scope_audit": rel(CARD / "target_scope_audit.csv"),
    "target_lift_audit": rel(CARD / "target_lift_audit.csv"),
    "bc_readiness_scores": rel(CARD / "bc_readiness_scores.csv"),
    "target_readiness_summary": rel(CARD / "target_readiness_summary.json"),
    "candidate_gap_by_fold": rel(CARD / "candidate_gap_by_fold.csv"),
    "global_candidate_objective_scores": rel(CARD / "global_candidate_objective_scores.csv"),
    "checkpoint_objective_scores": rel(CARD / "checkpoint_objective_scores.csv"),
    "selected_fold_teacher_metrics": rel(CARD / "selected_fold_teacher_metrics.csv"),
    "state_head_slice_precondition": rel(CARD / "state_head_slice_precondition.csv"),
    "precondition_gates": rel(CARD / "precondition_gates.json"),
    "objective_recipe": rel(CARD / "objective_recipe.json"),
    "objective_terms": rel(CARD / "objective_terms.csv"),
    "selection": rel(CARD / "selection.json"),
    "summary": rel(CARD / "summary.json"),
}

summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "next_mechanism": next_mechanism,
    "canonical_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "direct_comparison_baseline": "release16_risk_gated_state_head_objective_screen_v1",
    "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "source_card": "release16_risk_gated_state_head_objective_screen_v1",
    "source_path": rel(SOURCE_CARD / "summary.json"),
    "source_target_contract": "release16_failedfold_state_head_target_recipe_v1",
    "source_checkpoint_screen": "release16_failedfold_state_head_checkpoint_bc_screen_v1",
    "source_decision": source_summary.get("decision"),
    "role": "diagnose",
    "changed_layer": ["model_selection", "checkpoint_objective", "state_head_target_gating"],
    "target_window": REQUIRED_FOLDS,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "target_relative_return_by_window_pp": {fold: TARGET_RELATIVE_RETURN_PP for fold in REQUIRED_FOLDS},
    "drawdown_gate_operator": "<",
    "metrics": selected_metrics_frame.drop(columns=["candidate", "family", "candidate_gate"]).to_dict(orient="records"),
    "metric_basis": "diagnostic_per_fold_risk_gated_teacher_mix_not_retained_actor",
    "selected_by_fold": source_summary.get("selected_by_fold", {}),
    "failed_folds": FAILED_FOLDS,
    "gate_pass_count": int(source_summary.get("gate_pass_count", 0)),
    "worst_wf_max_drawdown_pct": float(selected_metrics_frame["max_dd_pct"].max()),
    "min_wf_relative_return_pp": float(selected_metrics_frame["relative_return_pct"].min()),
    "max_relative_gap_to_target_pp": float((TARGET_RELATIVE_RETURN_PP - selected_metrics_frame["relative_return_pct"]).clip(lower=0.0).max()),
    "max_drawdown_gap_to_target_pp": float((selected_metrics_frame["max_dd_pct"] - TARGET_MAX_DD_PCT).clip(lower=0.0).max()),
    "target_contract_gate": target_contract_gate,
    "bc_readiness_gate": bc_readiness_gate,
    "bc_screen_allowed_next": checkpoint_objective_allowed_next,
    "next_training_allowed_after_objective_screen": checkpoint_objective_allowed_next,
    "state_head_slice_precondition_gate": state_head_slice_precondition_gate,
    "ppo_allowed_next": ppo_allowed_next,
    "promotion_allowed": promotion_allowed,
    "actor_retention_allowed": actor_retention_allowed,
    "training_allowed_from_this_card": False,
    "primary_blocker": "training_contract_ready" if checkpoint_objective_allowed_next else "target_contract_blocked",
    "scope_summary": scope_summary,
    "lift_summary": lift_summary,
    "bc_readiness_summary": bc_readiness_summary,
    "checkpoint_selection_summary": checkpoint_selection_summary,
    "best_existing_single_candidate": best_existing_single_candidate,
    "existing_full_gate_candidate_count": int(checkpoint_objective_scores["window_gate"].astype(bool).sum()) if not checkpoint_objective_scores.empty else 0,
    "sidecars": sidecars,
    "summary_points": [
        "Existing global risk-gated candidates still have no full gate pass, so no actor can be retained from path selection.",
        f"Best existing single candidate is {best_existing_single_candidate.get('candidate', '')} with min relative {best_existing_single_candidate.get('min_relative_return_pp', 0.0):.2f}pp.",
        f"The per-fold teacher mix still only reaches min relative {float(selected_metrics_frame['relative_return_pct'].min()):.2f}pp with worst MaxDD {float(selected_metrics_frame['max_dd_pct'].max()):.2f}%.",
        f"Target scope/lift/BC readiness gates are {target_contract_gate}/{bc_readiness_gate}; state-head slice precondition is {state_head_slice_precondition_gate}.",
        "Next action is a nonconvex checkpoint/reward objective with slice guard, followed by official three-fold replay before any actor retention.",
    ],
}
selection = dict(summary)
selection["selection"] = {
    "selected_candidate": "nonconvex_risk_gated_state_head_checkpoint_objective",
    "selected_existing_single_candidate": best_existing_single_candidate.get("candidate", ""),
    "target_contract_gate": target_contract_gate,
    "bc_readiness_gate": bc_readiness_gate,
    "state_head_slice_precondition_gate": state_head_slice_precondition_gate,
    "checkpoint_objective_allowed_next": checkpoint_objective_allowed_next,
}

write_json(CARD / "summary.json", summary)
write_json(CARD / "selection.json", selection)

print({
    "decision": decision,
    "next_mechanism": next_mechanism,
    "frontier_gate_fields": {
        "target_contract_gate": target_contract_gate,
        "bc_readiness_gate": bc_readiness_gate,
        "bc_screen_allowed_next": checkpoint_objective_allowed_next,
    },
})

{'decision': 'release16_nonconvex_checkpoint_objective_readiness_ready_no_actor_retention', 'next_mechanism': 'train_nonconvex_risk_gated_state_head_checkpoint_objective_with_slice_guard_then_official_three_fold_replay', 'frontier_gate_fields': {'target_contract_gate': True, 'bc_readiness_gate': True, 'bc_screen_allowed_next': True}}


## Conclusion

This notebook does not retain an actor. It converts `release16_risk_gated_state_head_objective_screen_v1` into a training/checkpoint objective handoff.

Key result: target contract scope, target lift, and BC label readiness remain usable, but existing risk-gated path candidates still do not clear all three official walk-forward gates. The best single existing candidate is a diagnostic reference only; the next executable step is to train or score a nonconvex state-head checkpoint objective with a drawdown-drift stop and state-head slice guard, then rerun official `wf_2020`, `wf_2022`, and `wf_2023_2024` replay before any actor retention.